# Modelo hedónico de valoración inmobiliaria

Estima el **precio de mercado esperado** de una propiedad en Macul, La Florida, Ñuñoa y San Miguel,
a partir de sus atributos.

**Dataset:** `data/ofertas_unificado.sqlite`, tabla `ofertas` — 15.808 avisos scrapeados entre el
26/07 y el 10/08 de 2026.

Dos convenciones del dataset que hay que tener presentes desde el principio:

1. Las celdas vacías contienen el **texto `"SIN_DATO"`**, no `NULL`.
2. Todas las columnas están tipadas **`TEXT`** en SQLite, incluidos los números (salvo `detalle_ok`).

Si se ignoran, pandas lee todo como texto y los modelos fallan en silencio.

In [1]:
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

DB = Path("data/ofertas_unificado.sqlite")
SEMILLA = 42

print(f"pandas {pd.__version__} · numpy {np.__version__}")
print(f"base: {DB}  ({DB.stat().st_size / 1e6:.0f} MB)")

pandas 3.0.5 · numpy 2.5.2
base: data\ofertas_unificado.sqlite  (28 MB)


## 1. Carga

Se abre en modo **solo lectura** (`mode=ro`): el notebook nunca escribe en `data/`.

In [2]:
with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True, timeout=15) as con:
    crudo = pd.read_sql_query("SELECT * FROM ofertas", con)

print(f"{crudo.shape[0]:,} filas × {crudo.shape[1]} columnas")
crudo.head(3)

15,808 filas × 30 columnas


,sitio,id_aviso,url,operacion,tipo,comuna,barrio,direccion,titulo,precio_valor,precio_moneda,precio_clp,precio_uf,m2_util,m2_total,dormitorios,banos,estacionamientos,bodegas,antiguedad_anos,ano_construccion,gastos_comunes_clp,lat,lon,descripcion,publica,detalle_ok,fecha_scrape,antiguedad_aviso_dias,fecha_publicacion
0,pi,4161007982,https://portalinmobiliario.com/MLC-4161007982-...,venta,casa,nunoa,Parque Juan XXIII,"Los Talaveras 919, Ñuñoa, Parque Juan XXIII, Ñ...",Brown Sur Casa 2 Pisos Terreno 351 M2,9900.0,UF,404363421,9900.0,161,351,SIN_DATO,SIN_DATO,3,1,75,1951,0,-33.4641058,-70.5918346,Acogedora casa en tradicional barrio de Ñuñoa\...,SIN_DATO,1,2026-07-26,18,2026-07-08
1,pi,2016676099,https://portalinmobiliario.com/MLC-2016676099-...,venta,casa,nunoa,Plaza Ñuñoa,"Galicia Con Doctor Johow, Plaza Ñuñoa, Ñuñoa",Paraíso En Lo Mejor De Ñuñoa Con Gran Terreno,14000.0,UF,571827060,14000.0,356,467,SIN_DATO,SIN_DATO,SIN_DATO,1,81,1945,SIN_DATO,-33.4586524,-70.5935998,MAGNITUD EN NUNOA: 356 m2 Construidos y 445 m2...,SIN_DATO,1,2026-07-26,34,2026-06-22
2,pi,3804135116,https://portalinmobiliario.com/MLC-3804135116-...,venta,casa,nunoa,Parque Juan XXIII,"Av. Presidente Batlle Y Ordoñez, 3600 - 3900, ...","Residencial O Comercial, Vive El Estilo Art De...",13900.0,UF,567742581,13900.0,254,490,6,4,4,SIN_DATO,0,2026,SIN_DATO,-33.4495254,-70.5888929,Casa Ubicada en Diagonal Oriente con Pedro Tor...,SIN_DATO,1,2026-07-26,180,2026-01-27


## 2. Las dos convenciones, vistas en los datos

Antes de tocar nada, conviene ver el problema con los propios datos.

In [3]:
# Todo llega como texto: `precio_uf` no se puede promediar todavía.
print("dtypes distintos en el DataFrame:", dict(crudo.dtypes.value_counts()))
print(f"\nejemplo de precio_uf: {crudo['precio_uf'].head(3).tolist()}  <- son strings")

# Y los faltantes son el texto SIN_DATO, no NaN.
print(f"\nNaN reales en toda la tabla: {int(crudo.isna().sum().sum())}")

cobertura = (crudo != "SIN_DATO").mean().mul(100).sort_values()
print("\n% de filas CON dato, columnas peor cubiertas:")
print(cobertura.head(8).round(1).to_string())

dtypes distintos en el DataFrame: {<StringDtype(storage='python', na_value=nan)>: np.int64(29), dtype('int64'): np.int64(1)}

ejemplo de precio_uf: ['9900.0', '14000.0', '13900.0']  <- son strings

NaN reales en toda la tabla: 0

% de filas CON dato, columnas peor cubiertas:
publica                0.0
estacionamientos      52.7
gastos_comunes_clp    74.3
dormitorios           74.4
precio_clp            76.3
ano_construccion      77.3
antiguedad_anos       77.3
precio_uf             80.4


## 3. Los segmentos

`operacion` y `tipo` definen qué modelos son viables: los targets tienen escalas incomparables
(miles de UF en venta contra cientos de miles de CLP en arriendo) y ninguna propiedad está a la vez
en venta y en arriendo.

In [4]:
print(pd.crosstab(crudo["operacion"], crudo["tipo"], margins=True, margins_name="total"))
print(f"\npor fuente: {crudo['sitio'].value_counts().to_dict()}")

tipo       casa  departamento  total
operacion                           
arriendo    224          5611   5835
venta      3124          6849   9973
total      3348         12460  15808



por fuente: {'pi': 14462, 'cp': 1346}


---

## Siguiente paso

Convertir `"SIN_DATO"` a `NaN` y castear los números — en ese orden, porque hacerlo al revés impide
distinguir un faltante legítimo de un valor que no parsea.